# 1. Tiền xử lý dữ liệu: Lọc nhãn
Từ thư mục `data`, chúng ta chỉ giữ lại các nhãn: `helmet` (id=5), `person` (id=6), `vest` (id=7). Các nhãn khác sẽ bị loại bỏ. Dữ liệu mới được lưu vào `data_filtered`.

In [ ]:
import os
import shutil
import yaml

input_dir = 'data'
output_dir = 'data_filtered'

# Khai báo map ID cũ sang ID mới
# 5: helmet -> 0, 6: person -> 1, 7: vest -> 2
class_mapping = {5: 0, 6: 1, 7: 2}
new_names = ['helmet', 'person', 'vest']

# Tạo thư mục
for split in ['train', 'valid', 'test']:
    os.makedirs(os.path.join(output_dir, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(output_dir, split, 'labels'), exist_ok=True)
    
    img_dir = os.path.join(input_dir, split, 'images')
    lbl_dir = os.path.join(input_dir, split, 'labels')
    
    if not os.path.exists(img_dir): continue
        
    for filename in os.listdir(lbl_dir):
        if not filename.endswith('.txt'): continue
        
        lbl_path = os.path.join(lbl_dir, filename)
        new_lbl_path = os.path.join(output_dir, split, 'labels', filename)
        
        with open(lbl_path, 'r') as f:
            lines = f.readlines()
            
        new_lines = []
        for line in lines:
            parts = line.strip().split()
            if len(parts) > 0:
                cls_id = int(parts[0])
                if cls_id in class_mapping:
                    parts[0] = str(class_mapping[cls_id])
                    new_lines.append(' '.join(parts) + '\n')
                    
        # Nếu ảnh có nhãn hợp lệ, hoặc kể cả không có nhãn (ảnh nền), ta đều giữ lại cấu trúc
        with open(new_lbl_path, 'w') as f:
            f.writelines(new_lines)
            
        # Copy ảnh
        for ext in ['.jpg', '.png', '.jpeg']:
            img_filename = filename.replace('.txt', ext)
            if os.path.exists(os.path.join(img_dir, img_filename)):
                shutil.copy(os.path.join(img_dir, img_filename), os.path.join(output_dir, split, 'images', img_filename))
                break

# Tạo file data.yaml cho dataset mới
yaml_content = {
    'path': os.path.abspath(output_dir),
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': 3,
    'names': new_names
}
with open(os.path.join(output_dir, 'data.yaml'), 'w') as f:
    yaml.dump(yaml_content, f, sort_keys=False)

print("Hoàn tất lọc dữ liệu. Đã lưu dataset mới vào thư mục 'data_filtered'.")


# 2. Kiểm tra GPU
Huấn luyện mô hình YOLO đòi hỏi sức mạnh tính toán lớn, việc sử dụng GPU (Card màn hình) sẽ giúp tốc độ nhanh hơn CPU gấp nhiều lần.

In [1]:
import torch
import ultralytics

ultralytics.checks()

if torch.cuda.is_available():
    print(f"\nĐÃ TÌM THẤY GPU: {torch.cuda.get_device_name(0)}")
    print("Mô hình sẽ chạy trên GPU để tăng tốc tối đa.")
else:
    print("\nCẢNH BÁO: Không tìm thấy GPU CUDA. Quá trình huấn luyện bằng CPU sẽ RẤT CHẬM.")


Ultralytics 8.4.47  Python-3.10.11 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3060, 12288MiB)
Setup complete  (28 CPUs, 31.8 GB RAM, 209.0/244.0 GB disk)

ĐÃ TÌM THẤY GPU: NVIDIA GeForce RTX 3060
Mô hình sẽ chạy trên GPU để tăng tốc tối đa.


# 3. Huấn luyện mô hình YOLO
- Bắt buộc sử dụng GPU qua tham số `device=0`.
- Sử dụng chiến lược tăng cường dữ liệu (Data Augmentation) cơ bản để không làm chậm quá trình train.
- Lưu checkpoint mô hình mỗi 10 epoch.
- Tối ưu hóa đọc dữ liệu bằng cách sử dụng RAM cache, nhiều worker và Automatic Mixed Precision (amp).
- Epoch: 150
- Early stopping (patience): 20 (Dừng sớm nếu không cải thiện sau 20 epoch)
- Mục tiêu: Tối ưu để đạt mAP50 > 85%, mAP50-95 > 70%.

In [3]:
from ultralytics import YOLO

# Khởi tạo mô hình (ví dụ: dùng YOLOv8n làm pre-trained. Bạn có thể đổi sang bản cao hơn như yolov8s.pt nếu muốn mAP cao hơn)
model = YOLO('yolov8n.pt') 

# Tiến hành huấn luyện
results = model.train(
    data='data_filtered/data.yaml', 
    epochs=150, 
    patience=10,          # Dừng sớm nếu không tăng lên sau 20 epoch
    batch=16,             # Tăng batch size để tối ưu tốc độ (nếu GPU không đủ VRAM, hạ xuống 16 hoặc 8)
    imgsz=640,
    project='runs/train',
    name='ppe_model',
    
    # --- ÉP SỬ DỤNG GPU --- #
    device=0,             # Chạy trên Card màn hình rời (GPU số 0)
    
    # --- TỐI ƯU THỜI GIAN HUẤN LUYỆN --- #
    save_period=10,       # Lưu file weights mỗi 10 epoch
    cache=True,           # Lưu dữ liệu vào RAM (cache) để tăng tốc độ load dữ liệu
    workers=0,            # Số luồng CPU để xử lý dữ liệu (tuỳ thuộc vào số nhân CPU của bạn)
    amp=True,             # Bật Automatic Mixed Precision để tăng tốc độ tính toán trên GPU
    
    # --- CHIẾN LƯỢC TĂNG CƯỜNG DỮ LIỆU --- #
    hsv_h=0.015,          # Tinh chỉnh màu sắc
    hsv_s=0.7,            # Độ bão hòa
    hsv_v=0.4,            # Độ sáng
    degrees=10.0,         # Xoay ngẫu nhiên
    translate=0.1,        # Dịch chuyển ngẫu nhiên
    scale=0.5,            # Thu phóng ngẫu nhiên
    shear=2.0,            # Cắt nghiêng
    fliplr=0.5,           # Lật ngang (50% xác suất)
    mosaic=1.0,           # Trộn 4 ảnh thành 1 (hiệu quả cao, ít tốn kém)
    
    # --- CÁC THAM SỐ TỐI ƯU HÓA --- #
    lr0=0.01,             # Learning rate ban đầu
    lrf=0.01,             # Hệ số giảm learning rate ở epoch cuối
    optimizer='auto'      # Tự động chọn optimizer (SGD hoặc AdamW)
)

print("Hoàn tất huấn luyện mô hình.")


Ultralytics 8.4.47  Python-3.10.11 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3060, 12288MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data_filtered/data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=ppe_model-5, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, pa

SystemError: <built-in function warpAffine> returned a result with an exception set

In [4]:
from ultralytics import YOLO

# 1. Khởi tạo mô hình bằng file trọng số của lần chạy cuối cùng (last.pt) thay vì 'yolov8n.pt'
model = YOLO('runs/detect/runs/train/ppe_model-5/weights/last.pt')

# 2. Bật chế độ resume=True. 
# YOLO sẽ tự động đọc lại các thông số cũ (epochs=150, batch, data, v.v...) và chạy tiếp từ epoch 54.
results = model.train(resume=True)


Ultralytics 8.4.47  Python-3.10.11 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3060, 12288MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data_filtered\data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=runs\detect\runs\train\ppe_model-5\weights\last.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=ppe_model-5, nbs=64, nms=False, opset=None, optimize=Fals

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x000001C7772C8820>
Traceback (most recent call last):
  File "c:\Users\HaiK74\Desktop\VTV_2_UTT\venv\lib\site-packages\torch\utils\data\dataloader.py", line 1604, in __del__
    self._shutdown_workers()
  File "c:\Users\HaiK74\Desktop\VTV_2_UTT\venv\lib\site-packages\torch\utils\data\dataloader.py", line 1562, in _shutdown_workers
    if self._persistent_workers or self._workers_status[worker_id]:
AttributeError: '_MultiProcessingDataLoaderIter' object has no attribute '_workers_status'


Model summary: 130 layers, 3,011,433 parameters, 3,011,417 gradients, 8.2 GFLOPs

Transferred 355/355 items from pretrained weights
Freezing layer 'model.22.dfl.conv.weight'
AMP: running Automatic Mixed Precision (AMP) checks...
AMP: checks passed 
train: Fast image access  (ping: 0.10.0 ms, read: 43.335.4 MB/s, size: 52.0 KB)
train: Scanning C:\Users\HaiK74\Desktop\VTV_2_UTT\data_filtered\train\labels.cache... 9545 images, 562 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 9545/9545  0.0s
WARNING train: 16.4GB RAM required to cache images with 50% safety margin but only 8.0/31.8GB available, not caching images
val: Fast image access  (ping: 0.00.0 ms, read: 82.69.4 MB/s, size: 54.6 KB)
val: Scanning C:\Users\HaiK74\Desktop\VTV_2_UTT\data_filtered\valid\labels.cache... 2730 images, 169 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2730/2730  0.0s
WARNING cache='ram' may produce non-deterministic training results. Consider cache='disk' as a deterministic alternative if your disk space allows.
va